# Experiment 2 — jobs, stages, tasks of a small pipeline

Spark UI: http://localhost:4040 → **Jobs**, **Stages**, **SQL / DataFrame**.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m1-exp2-pipeline").getOrCreate()

orders = spark.read.parquet("/opt/data/m1/orders")
customers = spark.read.parquet("/opt/data/m1/customers")
print("orders files:", orders.rdd.getNumPartitions(), "| customers files:", customers.rdd.getNumPartitions())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 11:27:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

orders files: 8 | customers files: 1


## The pipeline: filter → groupBy → join → write

Defining it runs nothing (Experiment 1). Read it, then **PREDICT** in chat, before the write cell:

1. How many **jobs** will the write start?
2. How many **stages**, and what separates them?
3. Roughly how many **tasks** in each stage?
4. Where will `Exchange` appear in the plan?
5. Which join strategy do you expect, and does it add a shuffle?

In [2]:
result = (
    orders.filter(F.col("order_date") >= "2026-06-01")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total"), F.count("*").alias("n"))
    .join(customers, "customer_id")
    .select("customer_id", "country", "segment", "total", "n")
)

## Run the write (after predicting)

In [3]:
result.write.mode("overwrite").parquet("/opt/data/m1/out/customer_totals")

In [7]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

## Observe

- **Jobs** tab: how many jobs did the write create? Open each one and look at its DAG.
- **Stages** tab: tasks per stage; which stages show *Shuffle Write* and which show *Shuffle Read*?
- **SQL / DataFrame** tab: open the query; find every `Exchange` and the join node.

Then cross-check against the plan:

In [4]:
result.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- BroadcastHashJoin Inner BuildRight (10)
      :- HashAggregate (6)
      :  +- Exchange (5)
      :     +- HashAggregate (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan parquet  (1)
      +- BroadcastExchange (9)
         +- Filter (8)
            +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [customer_id#1, amount#3, order_date#4]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(order_date), GreaterThanOrEqual(order_date,2026-06-01), IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,amount:double,order_date:date>

(2) Filter
Input [3]: [customer_id#1, amount#3, order_date#4]
Condition : ((isnotnull(order_date#4) AND (order_date#4 >= 2026-06-01)) AND isnotnull(customer_id#1))

(3) Project
Output [2]: [customer_id#1, amount#3]
Input [3]: [customer_id#1, amount#3, order_date#4]

(4) HashAggregate
Input [2]: [customer

In [6]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
result.write.mode("overwrite").parquet("/opt/data/m1/out/customer_totals_noaqe")

## Variant — change one thing

A second aggregation on the same key, on top of `result`.

**PREDICT:** how many `Exchange` nodes in this plan compared with the one above?

In [8]:
result2 = result.groupBy("customer_id").agg(F.max("total").alias("max_total"))
result2.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- HashAggregate (13)
   +- HashAggregate (12)
      +- Project (11)
         +- BroadcastHashJoin Inner BuildRight (10)
            :- HashAggregate (6)
            :  +- Exchange (5)
            :     +- HashAggregate (4)
            :        +- Project (3)
            :           +- Filter (2)
            :              +- Scan parquet  (1)
            +- BroadcastExchange (9)
               +- Filter (8)
                  +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [customer_id#1, amount#3, order_date#4]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/orders]
PushedFilters: [IsNotNull(order_date), GreaterThanOrEqual(order_date,2026-06-01), IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,amount:double,order_date:date>

(2) Filter
Input [3]: [customer_id#1, amount#3, order_date#4]
Condition : ((isnotnull(order_date#4) AND (order_date#4 >= 2026-06-01)) AND isnotnull(customer_id#1))

(3) Project
Output [2]: [cu

## Stop before opening the next notebook

In [ ]:
spark.stop()